In [2]:
import pandas as pd
# Import raw dataset
raw_df = pd.read_csv('superstore\\raw\\sample_superstore.csv')

# Small Caps and Space replacing with underscore
raw_df.columns = [col.lower().replace(' ', '_').replace('-', '_') for col in raw_df.columns]
# Convert date to datetime format and normal format
raw_df['order_date'] = pd.to_datetime(raw_df['order_date'])
raw_df['ship_date'] = pd.to_datetime(raw_df['ship_date'])

print(raw_df)

      row_id        order_id order_date  ship_date       ship_mode  \
0          1  CA-2016-152156 2016-11-08 2016-11-11    Second Class   
1          2  CA-2016-152156 2016-11-08 2016-11-11    Second Class   
2          3  CA-2016-138688 2016-06-12 2016-06-16    Second Class   
3          4  US-2015-108966 2015-10-11 2015-10-18  Standard Class   
4          5  US-2015-108966 2015-10-11 2015-10-18  Standard Class   
...      ...             ...        ...        ...             ...   
9989    9990  CA-2014-110422 2014-01-21 2014-01-23    Second Class   
9990    9991  CA-2017-121258 2017-02-26 2017-03-03  Standard Class   
9991    9992  CA-2017-121258 2017-02-26 2017-03-03  Standard Class   
9992    9993  CA-2017-121258 2017-02-26 2017-03-03  Standard Class   
9993    9994  CA-2017-119914 2017-05-04 2017-05-09    Second Class   

     customer_id     customer_name    segment        country             city  \
0       CG-12520       Claire Gute   Consumer  United States        Henderson 

In [4]:
# Select numeric columns for correlation
# Correlation values range from -1 to 1 (1 is a perfect positive relationship)
numeric_cols = raw_df[['sales', 'quantity', 'discount', 'profit']]
correlation_matrix = numeric_cols.corr()
print("--- Correlation Matrix (Profit Drivers) ---")
print(correlation_matrix['profit'].sort_values(ascending=False))

--- Correlation Matrix (Profit Drivers) ---
profit      1.000000
sales       0.479064
quantity    0.066253
discount   -0.219487
Name: profit, dtype: float64


In [6]:
# Group by discount level and calculate average profit
discount_impact = raw_df.groupby('discount').agg(
    avg_profit=('profit', 'mean'),
    total_orders=('order_id', 'count')).reset_index()
# Identify the 'Loss Threshold'
print("--- Average Profit by Discount Level ---")
print(discount_impact.sort_values(by='discount').round(2))

--- Average Profit by Discount Level ---
    discount  avg_profit  total_orders
0       0.00       66.90          4798
1       0.10       96.06            94
2       0.15       27.29            52
3       0.20       24.70          3657
4       0.30      -45.68           227
5       0.32      -88.56            27
6       0.40     -111.93           206
7       0.45     -226.65            11
8       0.50     -310.70            66
9       0.60      -43.08           138
10      0.70      -95.87           418
11      0.80     -101.80           300


In [8]:
# Extract the month name
raw_df['month_name'] = raw_df['order_date'].dt.month_name()
# Average sales per month across all years
seasonal_trends = raw_df.groupby('month_name')['sales'].mean().round(2).reset_index()
# Sort by calendar order (not alphabetical)
months_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                'July', 'August', 'September', 'October', 'November', 'December']
seasonal_trends['month_name'] = pd.Categorical(seasonal_trends['month_name'], categories=months_order, ordered=True)
seasonal_trends = seasonal_trends.sort_values('month_name')

print("--- Seasonal Sales Trend (Average per Month) ---")
print(seasonal_trends)

--- Seasonal Sales Trend (Average per Month) ---
   month_name   sales
4     January  249.15
3    February  199.17
7       March  294.55
0       April  206.23
8         May  210.92
6        June  213.00
5        July  207.38
1      August  225.27
11  September  222.45
10    October  244.59
9    November  239.61
2    December  231.03


In [10]:
# Regional Profitability Scorecard 
region_check = raw_df.groupby('region').agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    avg_discount=('discount', 'mean')).reset_index()

region_check['profit_margin'] = (region_check['total_profit'] / region_check['total_sales']) * 100
print("--- 1. Regional Profitability Check ---")
print(region_check.sort_values(by='profit_margin').round(2))

--- 1. Regional Profitability Check ---
    region  total_sales  total_profit  avg_discount  profit_margin
0  Central    501239.89      39706.36          0.24           7.92
2    South    391721.90      46749.43          0.15          11.93
1     East    678781.24      91522.78          0.15          13.48
3     West    725457.82     108418.45          0.11          14.94


In [12]:
# Category Mix Analysis 
category_check = raw_df.groupby('category').agg(
    total_profit=('profit', 'sum'),
    profit_margin=('profit', 'sum'), # Placeholder for calculation
    total_sales=('sales', 'sum')).reset_index()

category_check['profit_margin'] = (category_check['total_profit'] / category_check['total_sales']) * 100
print("--- 2. Category Performance ---")
print(category_check.sort_values(by='profit_margin').round(2))

--- 2. Category Performance ---
          category  total_profit  profit_margin  total_sales
0        Furniture      18451.27           2.49    741999.80
1  Office Supplies     122490.80          17.04    719047.03
2       Technology     145454.95          17.40    836154.03


In [16]:
# The "Discount Trap" Correlation 
# Define discount bins (0%, 0-20%, 20-50%, 50%+)
bins = [0, 0.001, 0.21, 0.51, 1.0]
labels = ['No Discount', 'Low (0-20%)', 'Medium (20-50%)', 'High (50%+)']

raw_df['discount_bracket'] = pd.cut(raw_df['discount'], bins=bins, labels=labels, include_lowest=True)

discount_impact = raw_df.groupby('discount_bracket', observed=False).agg(
    avg_profit=('profit', 'mean'),
    order_count=('order_id', 'count')).reset_index()

print("--- 3. Discount Impact Analysis ---")
print(discount_impact.round(2))

--- 3. Discount Impact Analysis ---
  discount_bracket  avg_profit  order_count
0      No Discount       66.90         4798
1      Low (0-20%)       26.50         3803
2  Medium (20-50%)     -109.53          537
3      High (50%+)      -89.44          856


In [22]:
# Finding the exact location of the problem 
problem_segments = raw_df.groupby(['region', 'category']).agg(
    avg_discount=('discount', 'mean'),
    total_profit=('profit', 'sum')).reset_index()

# Filter for the worst performing intersections 
worst_performers = problem_segments[problem_segments['total_profit'] < 0].sort_values(by='total_profit')

print("\n--- Final Root Cause Identification ---")
print(worst_performers.round(2))

# Create a pivot table for Region vs Category
pivot_profit = raw_df.pivot_table(
    index='region', 
    columns='category', 
    values='profit', 
    aggfunc='sum')

print("--- Regional Profit Heatmap Data ---")
print(pivot_profit.round(2))


--- Final Root Cause Identification ---
    region   category  avg_discount  total_profit
0  Central  Furniture           0.3      -2871.05

--- Regional Profit Heatmap Data ---
category  Furniture  Office Supplies  Technology
region                                          
Central    -2871.05          8879.98    33697.43
East        3046.17         41014.58    47462.04
South       6771.21         19986.39    19991.83
West       11504.95         52609.85    44303.65


In [31]:
# Forecast Next Month Sales 
# Get monthly sales totals
raw_df['month_year'] = raw_df['order_date'].dt.to_period('M')
monthly_sales = raw_df.groupby('month_year')['sales'].sum().reset_index()

# Calculate Month-over-Month growth rate
monthly_sales['growth_rate'] = monthly_sales['sales'].pct_change()

# Get the average growth rate of the last 3 months
avg_growth = monthly_sales['growth_rate'].tail(3).mean()

# Forecast: Last month's sales * (1 + average growth)
last_month_sales = monthly_sales['sales'].iloc[-1]
forecast_next_month = last_month_sales * (1 + avg_growth)

print(f"Average Growth Rate (Last 3 months): {avg_growth:.2%}")
print(f"Forecasted Sales for Next Month: ${forecast_next_month:,.2f}")

Average Growth Rate (Last 3 months): 3.86%
Forecasted Sales for Next Month: $87,065.67


In [34]:
# Identify High-Risk Customers 
# Calculate Recency and Frequency per customer
current_date = raw_df['order_date'].max() # Using the latest date in the dataset as 'today'

customer_risk = raw_df.groupby(['customer_id', 'customer_name']).agg(
    last_purchase=('order_date', 'max'),
    order_count=('order_id', 'nunique')).reset_index()

# Calculate days since last purchase
customer_risk['days_since_last'] = (current_date - customer_risk['last_purchase']).dt.days

# Define the Risk Logic
# High Risk = Only 1 order AND hasn't bought in 6+ months (180 days)
high_risk = customer_risk[
    (customer_risk['order_count'] == 1) & 
    (customer_risk['days_since_last'] > 180)]

print(f"Total High-Risk Customers identified: {len(high_risk)}")
print(high_risk.sort_values('days_since_last', ascending=False).head(10))

Total High-Risk Customers identified: 8
    customer_id      customer_name last_purchase  order_count  days_since_last
637    RE-19405    Ricardo Emerson    2014-12-29            1             1097
701    SM-20905  Susan MacKendrick    2016-05-03            1              606
456    LD-16855       Lela Donovan    2016-06-26            1              552
49     AR-10570     Anemone Ratner    2016-07-14            1              534
42     AO-10810  Anthony O'Donnell    2016-08-16            1              501
145    CJ-11875       Carl Jackson    2016-12-30            1              365
650    RM-19750      Roland Murray    2017-03-06            1              299
508    MG-18205    Mitch Gastineau    2017-04-10            1              264
